# 01 - Môi Trường, Cấu Hình Và Mô Hình Nền

Notebook này không phải nơi chatbot trả lời khách. Đây là **phòng máy** của hệ thống: kiểm tra đường dẫn, dịch vụ nền, tên model, checkpoint và các lớp embedding.

Một người mới thường nhìn RAG như một chatbot. Ở notebook này, hãy đổi góc nhìn: hệ thống thật ra là một dây chuyền biến dữ liệu thành vector, rồi dùng vector đó để tìm bằng chứng trước khi LLM viết câu trả lời.

```text
Dữ liệu / câu hỏi / ảnh
  -> embedding model biến thành vector
  -> Qdrant tìm vector gần nhất
  -> retriever/reranker chọn bằng chứng
  -> LLM chỉ viết dựa trên bằng chứng đó
```

**Nên chạy notebook này đầu tiên để kiểm tra**

- Python executable và môi trường ảo có đúng không.
- GPU/CPU PyTorch đang dùng gì.
- `CHATBOT_FASHION_DIR` và checkpoint ViFashionCLIP có tồn tại không.
- Qdrant, Ollama, ViFashionCLIP embedding service có đang bật không.
- Ba không gian vector có tách biệt rõ không: sản phẩm text 512, ảnh 512, rule phối đồ 1024.

> Output cũ đã được xoá. Khi bạn chạy lại, hãy tin output mới hơn là output từng lưu trong notebook gốc.


# Bản Đồ Tư Duy Của Fashion RAG Chatbot

Hãy xem hệ thống như một cửa hàng có ba loại trí nhớ khác nhau, chứ không phải một LLM duy nhất:

| Tầng | Câu hỏi mà tầng này trả lời | Model embedding | Vector | Nơi dùng |
|---|---|---|---:|---|
| Layer A text | “Sản phẩm nào trong kho giống nhu cầu text này?” | `ViFashionCLIPTextEmbeddings` | 512 | Tìm sản phẩm bằng tiếng Việt |
| Layer A image | “Ảnh này giống sản phẩm nào trong kho?” | `FashionCLIPImageEmbeddings` | 512 | Tìm sản phẩm bằng ảnh |
| Layer B rule | “Theo stylist, nên phối kiểu gì?” | `BGEM3Embeddings` | 1024 | Tìm quy tắc phối đồ |

Điểm quan trọng: **Layer A và Layer B không được trộn vector với nhau**. Dù cùng gọi là embedding, chúng đang sống trong các không gian hình học khác nhau. Vector 512 của sản phẩm không thể đem tìm trong collection 1024 của rule phối đồ.

## Luồng Tổng Quát

```text
User gửi text hoặc ảnh
  -> kiểm tra input có an toàn không
  -> nếu có ảnh: phân loại ảnh sản phẩm hay ảnh người
  -> router chọn luồng: tìm sản phẩm, tìm bằng ảnh, phối đồ, hỏi profile, chào hỏi
  -> retrieval lấy bằng chứng từ Qdrant
  -> LLM viết câu trả lời dựa trên bằng chứng
  -> guardrail kiểm tra LLM có bịa mã sản phẩm không
```

Notebook 01 chỉ chuẩn bị “ngôn ngữ hình học” cho các tầng sau. Nếu notebook này sai, những notebook debug retrieval phía sau sẽ sai theo.


## BƯỚC 1: Kiểm Tra Môi Trường

Cell này trả lời câu hỏi rất thực tế: notebook đang chạy bằng Python nào?

Nếu Python executable không nằm trong môi trường bạn cài dependency cho dự án, các lỗi phía sau sẽ rất khó hiểu: import thiếu thư viện, không thấy GPU, hoặc model load chậm bất thường.


In [1]:
import sys

print(f"Python executable: {sys.executable}")
print(f"Python version   : {sys.version}")
print("\nGợi ý kiểm tra:")
print("- Nếu dùng venv, đường dẫn nên chứa thư mục venv của project.")
print("- Nếu dùng conda, đường dẫn nên thuộc env bạn đã cài torch/langchain/qdrant.")


Python executable: c:\Users\DELL\anaconda3\python.exe
Python version   : 3.12.4 | packaged by Anaconda, Inc. | (main, Jun 18 2024, 15:03:56) [MSC v.1929 64 bit (AMD64)]

Gợi ý kiểm tra:
- Nếu dùng venv, đường dẫn nên chứa thư mục venv của project.
- Nếu dùng conda, đường dẫn nên thuộc env bạn đã cài torch/langchain/qdrant.


## BƯỚC 2: Import Thư Viện Cần Cho Notebook Này

Notebook gốc import gần như toàn bộ hệ thống ở một chỗ. Với notebook nền này, mình chỉ giữ những import thật sự cần để hiểu và định nghĩa các model embedding.

Đã bỏ khỏi notebook này các import gây nhiễu như chain, prompt, Redis history, Qdrant client, intent router. Chúng thuộc các notebook sau:

- Qdrant/retriever: notebook 03, 04, 05
- Prompt/chain/history: notebook 07
- Router/security/eval: notebook 08

Cách nghĩ mới: **import càng ít, lỗi càng dễ khoanh vùng**.


In [2]:
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from langchain_core.embeddings import Embeddings
from langchain_ollama import OllamaEmbeddings
from PIL import Image
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer, CLIPModel, CLIPProcessor

print("[OK] Import tối thiểu cho notebook 01 đã hoàn tất")
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device     : {torch.cuda.get_device_name(0)}")
else:
    print("CUDA device     : CPU only")


[OK] Import tối thiểu cho notebook 01 đã hoàn tất
PyTorch version : 2.7.0+cpu
CUDA available  : False
CUDA device     : CPU only


## BƯỚC 3: Constants Và Cấu Hình Mô Hình

Đây là “bảng điện” của hệ thống. Trước khi debug retrieval, hãy biết rõ từng model/collection đang được dùng để làm gì.

| Nhóm | Constant | Ý nghĩa |
|---|---|---|
| LLM | `LLM_MODEL` | Model viết câu trả lời, route fallback, summarize |
| Vision | `QWEN_VL_MODEL` | Model phân tích ảnh người/sản phẩm |
| Layer A text | `TEACHER_MODEL_NAME`, `STUDENT_MODEL_NAME` | Teacher FashionCLIP và student tiếng Việt |
| Qdrant | `PRODUCT_COLLECTION`, `PRODUCT_IMAGE_COLLECTION`, `LAYER_B_*` | Các collection tách riêng theo loại vector |
| Retrieval | `PRODUCT_SEARCH_CANDIDATE_K`, `RERANKER_TOP_N` | Số ứng viên lấy ra trước/sau rerank |

Lưu ý: `STUDENT_MODEL_NAME` chỉ là giá trị fallback. Khi load checkpoint, notebook sẽ ưu tiên `student_model_name` được lưu trong checkpoint nếu có.


In [4]:
# ════════════════════════════════════════════════════════════════
# Service endpoints
# ════════════════════════════════════════════════════════════════
OLLAMA_BASE_URL = "http://localhost:11434"
QDRANT_URL = "http://localhost:6333"
VIFASHIONCLIP_SERVICE_URL = "http://localhost:18080"

# ════════════════════════════════════════════════════════════════
# LLM & Vision-Language models chạy local qua Ollama
# ════════════════════════════════════════════════════════════════
LLM_MODEL = "qwen3:4b-instruct"       # Chat chính, router fallback, summarize
QWEN_VL_MODEL = "qwen2.5vl:3b"        # Phân tích ảnh người/sản phẩm

# ════════════════════════════════════════════════════════════════
# ViFashionCLIP model names dùng cho Layer A text retrieval
# Teacher tạo không gian FashionCLIP 512 chiều.
# Student tiếng Việt học cách đi vào không gian đó.
# ════════════════════════════════════════════════════════════════
TEACHER_MODEL_NAME = "patrickjohncyh/fashion-clip"
STUDENT_MODEL_NAME = "AITeamVN/Vietnamese_Embedding_v2"
STUDENT_MAX_LENGTH = 128

# ════════════════════════════════════════════════════════════════
# ProjectionHead: map student_hidden_dim -> FashionCLIP 512-dim
# ════════════════════════════════════════════════════════════════
PROJECTION_HIDDEN_DIM = 1024
PROJECTION_NUM_LAYERS = 3
PROJECTION_DROPOUT = 0.05

# ════════════════════════════════════════════════════════════════
# Qdrant collections và kích thước vector kỳ vọng
# ════════════════════════════════════════════════════════════════
PRODUCT_COLLECTION = "fashion_products_vifashionclip_vi_65k_structured_vi"
PRODUCT_IMAGE_COLLECTION = "fashion_products_fashionclip_image_main_65k"
LAYER_B_FEMALE_COLLECTION = "layer_b_female"
LAYER_B_MALE_COLLECTION = "layer_b_male"

PRODUCT_VECTOR_SIZE = 512
IMAGE_VECTOR_SIZE = 512
LAYER_B_VECTOR_SIZE = 1024

# ════════════════════════════════════════════════════════════════
# Retrieval knobs: các số này ảnh hưởng trực tiếp đến tốc độ và chất lượng
# ════════════════════════════════════════════════════════════════
PRODUCT_SEARCH_CANDIDATE_K = 15  # Qdrant lấy raw candidates
RERANKER_TOP_N = 8               # Reranker chấm lại top-N candidates
PRODUCT_SEARCH_PAGE_SIZE = 5      # Số sản phẩm tối đa gửi vào LLM
PRODUCT_SEARCH_BRAND_LIMIT = 2    # Giảm lặp quá nhiều sản phẩm cùng brand

print("[OK] Cấu hình chính")
print(f"LLM                 : {LLM_MODEL}")
print(f"Vision              : {QWEN_VL_MODEL}")
print(f"Layer A text model  : {STUDENT_MODEL_NAME} -> FashionCLIP 512-dim")
print(f"Layer A text store  : {PRODUCT_COLLECTION} ({PRODUCT_VECTOR_SIZE}-dim)")
print(f"Layer A image store : {PRODUCT_IMAGE_COLLECTION} ({IMAGE_VECTOR_SIZE}-dim)")
print(f"Layer B stores      : {LAYER_B_FEMALE_COLLECTION}, {LAYER_B_MALE_COLLECTION} ({LAYER_B_VECTOR_SIZE}-dim)")
print(f"Retrieval           : top-{PRODUCT_SEARCH_CANDIDATE_K} -> rerank top-{RERANKER_TOP_N} -> final {PRODUCT_SEARCH_PAGE_SIZE}")


[OK] Cấu hình chính
LLM                 : qwen3:4b-instruct
Vision              : qwen2.5vl:3b
Layer A text model  : AITeamVN/Vietnamese_Embedding_v2 -> FashionCLIP 512-dim
Layer A text store  : fashion_products_vifashionclip_vi_65k_structured_vi (512-dim)
Layer A image store : fashion_products_fashionclip_image_main_65k (512-dim)
Layer B stores      : layer_b_female, layer_b_male (1024-dim)
Retrieval           : top-15 -> rerank top-8 -> final 5


## BƯỚC 4: Tìm Thư Mục Dự Án Và Checkpoint

`resolve_chatbot_fashion_dir()` giúp notebook chạy được dù bạn mở từ workspace root, từ `Chatbot_Fashion/`, hay từ thư mục notebook.

Checkpoint ViFashionCLIP là mảnh rất quan trọng: nó chứa trọng số encoder tiếng Việt và projection head đã train. Nếu checkpoint không tồn tại, Layer A text retrieval không thể encode query tiếng Việt theo đúng không gian FashionCLIP.


In [5]:
def resolve_chatbot_fashion_dir() -> Path:
    """
    Tìm thư mục gốc Chatbot_Fashion/ từ vị trí hiện tại.

    Vì notebook có thể được mở từ nhiều nơi khác nhau, hàm này thử theo thứ tự:
    1. Nếu đang ở thư mục notebooks/ thì đi lên thư mục cha.
    2. Nếu đang ở Chatbot_Fashion/ thì dùng luôn thư mục hiện tại.
    3. Nếu đang ở workspace root thì tìm thư mục con Chatbot_Fashion/.
    4. Nếu vẫn chưa thấy, đi dần qua các thư mục cha để tìm.
    """
    cwd = Path.cwd()

    if cwd.name == "notebooks" and cwd.parent.name == "Chatbot_Fashion":
        return cwd.parent
    if cwd.name == "Chatbot_Fashion":
        return cwd

    candidate = cwd / "Chatbot_Fashion"
    if candidate.exists():
        return candidate

    for parent in cwd.parents:
        candidate = parent / "Chatbot_Fashion"
        if candidate.exists():
            return candidate

    raise FileNotFoundError("Không tìm thấy Chatbot_Fashion/ từ thư mục hiện tại")


CHATBOT_FASHION_DIR = resolve_chatbot_fashion_dir()
VIFASHIONCLIP_CHECKPOINT = (
    CHATBOT_FASHION_DIR
    / "Vietnamese"
    / "vifashionclip_aiteamvn_embedding_v2_projection_336k"
    / "stage2_last_layers"
    / "best_stage2_model.pt"
)

print(f"[OK] Thư mục dự án      : {CHATBOT_FASHION_DIR}")
print(f"[OK] Checkpoint path    : {VIFASHIONCLIP_CHECKPOINT}")
print(f"     Checkpoint exists  : {VIFASHIONCLIP_CHECKPOINT.exists()}")

if not VIFASHIONCLIP_CHECKPOINT.exists():
    print("[CẢNH BÁO] Chưa thấy checkpoint. Layer A text embedding sẽ không load được.")


[OK] Thư mục dự án      : d:\KHÓA LUẬN\WORKSPACE\Chatbot_Fashion
[OK] Checkpoint path    : d:\KHÓA LUẬN\WORKSPACE\Chatbot_Fashion\Vietnamese\vifashionclip_aiteamvn_embedding_v2_projection_336k\stage2_last_layers\best_stage2_model.pt
     Checkpoint exists  : True


## BƯỚC 5: Kiểm Tra Nhanh Các Dịch Vụ Nền

Ba dịch vụ này thường là nguyên nhân làm notebook “chậm” hoặc treo:

- Qdrant ở `localhost:6333`: lưu vector và trả kết quả retrieval.
- Ollama ở `localhost:11434`: chạy LLM, VLM, BGE-M3.
- ViFashionCLIP service ở `localhost:18080`: encode query tiếng Việt cho Layer A nếu dùng backend remote.

Cell này chỉ ping nhẹ bằng HTTP timeout ngắn. Nếu một service fail, hãy sửa service trước khi nghi retrieval sai.


In [7]:
from urllib.error import URLError
from urllib.request import urlopen


def ping_http(name: str, url: str, timeout: float = 2.0) -> bool:
    """Ping một HTTP endpoint và in kết quả gọn, không làm notebook crash nếu service tắt."""
    try:
        with urlopen(url, timeout=timeout) as response:
            status = getattr(response, "status", "unknown")
        print(f"[OK] {name:<24} {url} -> HTTP {status}")
        return True
    except URLError as exc:
        print(f"[FAIL] {name:<22} {url} -> {exc}")
        return False
    except Exception as exc:
        print(f"[FAIL] {name:<22} {url} -> {type(exc).__name__}: {exc}")
        return False


service_status = {
    "qdrant": ping_http("Qdrant", f"{QDRANT_URL}/collections"),
    "ollama": ping_http("Ollama", f"{OLLAMA_BASE_URL}/api/tags"),
    "vifashionclip_service": ping_http("ViFashionCLIP service", f"{VIFASHIONCLIP_SERVICE_URL}/health"),
}

print("\nTóm tắt service:", service_status)


[OK] Qdrant                   http://localhost:6333/collections -> HTTP 200
[OK] Ollama                   http://localhost:11434/api/tags -> HTTP 200
[OK] ViFashionCLIP service    http://localhost:18080/health -> HTTP 200

Tóm tắt service: {'qdrant': True, 'ollama': True, 'vifashionclip_service': True}


## BƯỚC 6: Các Khối Neural Network Của ViFashionCLIP

Đây là phần dễ làm người mới bị ngợp, nhưng thật ra chỉ có một ý chính:

> Student encoder hiểu tiếng Việt, còn FashionCLIP hiểu thời trang/ảnh. ProjectionHead là “bộ chuyển giọng” để embedding tiếng Việt đi vào cùng không gian 512 chiều với FashionCLIP.

```text
Câu tiếng Việt
  -> tokenizer
  -> student encoder tạo embedding cho từng token
  -> mean_pool gom nhiều token thành một vector câu
  -> projection head map sang 512 chiều
  -> L2 normalize để cosine search ổn định hơn
```

Các class bên dưới chỉ định nghĩa kiến trúc. Chúng chưa load model nặng cho tới khi bạn khởi tạo `ViFashionCLIPTextEmbeddings`.


### Cách Đọc Bốn Khối Bên Dưới

Đọc theo thứ tự này:

1. `ResidualMLPBlock`: một block nhỏ giúp mạng học biến đổi sâu nhưng vẫn giữ tín hiệu gốc.
2. `ProjectionHead`: xếp nhiều block để đổi chiều từ hidden size của student sang 512 chiều.
3. `mean_pool`: gom embedding của từng token thành embedding của cả câu.
4. `Stage2StudentProjection`: ghép encoder + pooling + projection thành một model inference hoàn chỉnh.

Nếu chỉ debug retrieval, bạn không cần sửa các class này. Bạn chỉ cần kiểm tra chúng load đúng checkpoint và trả vector đúng chiều.


**`ResidualMLPBlock` - giữ tín hiệu gốc khi mạng học biến đổi**

Input và output có cùng shape, thường là `[batch_size, hidden_dim]`. Block này học phần “cần chỉnh thêm”, rồi cộng lại với input ban đầu. Cách này giúp projection head sâu hơn mà không làm mất tín hiệu gốc quá nhanh.


In [8]:
class ResidualMLPBlock(nn.Module):
    """
    Một MLP block có residual connection.

    Input:
        x: tensor shape [batch_size, dim]

    Output:
        tensor shape [batch_size, dim]

    Vì sao cần residual?
        ProjectionHead phải học cách biến embedding tiếng Việt sang không gian FashionCLIP.
        Nếu chỉ xếp nhiều Linear layer, tín hiệu gốc có thể bị biến dạng mạnh.
        Công thức x + block(x) cho phép model giữ lại phần hữu ích từ input,
        đồng thời học thêm phần hiệu chỉnh cần thiết.
    """

    def __init__(self, dim: int, dropout: float = 0.05):
        super().__init__()
        self.block = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim * 2),   # Mở rộng chiều để học tương tác phong phú hơn
            nn.GELU(),                 # Activation mượt, thường dùng trong transformer
            nn.Dropout(dropout),       # Giảm overfit khi training
            nn.Linear(dim * 2, dim),   # Thu về đúng chiều ban đầu để cộng residual
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return x + self.block(x)


**`ProjectionHead` - cầu nối từ tiếng Việt sang không gian FashionCLIP**

Student encoder có hidden size riêng, còn Qdrant collection sản phẩm đang lưu vector 512 chiều theo không gian FashionCLIP. Projection head chịu trách nhiệm map từ hidden size đó sang 512 chiều.


In [9]:
class ProjectionHead(nn.Module):
    """
    Ánh xạ embedding của student encoder sang không gian FashionCLIP 512 chiều.

    Tham số quan trọng:
        input_dim: hidden size của Vietnamese student encoder.
        output_dim: thường là 512, lấy từ checkpoint teacher_dim.
        hidden_dim: chiều trung gian của MLP.
        num_layers: số lớp biến đổi. Với 3 layer, sẽ có 1 ResidualMLPBlock ở giữa.

    Góc nhìn retrieval:
        Qdrant không hiểu text, nó chỉ so sánh vector.
        ProjectionHead quyết định vector tiếng Việt có nằm đúng “tọa độ thời trang” hay không.
    """

    def __init__(self, input_dim, output_dim, hidden_dim=1024, num_layers=3, dropout=0.05):
        super().__init__()
        layers = [
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        ]

        for _ in range(max(0, num_layers - 2)):
            layers.append(ResidualMLPBlock(hidden_dim, dropout=dropout))

        layers.extend([
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, output_dim),  # Output kỳ vọng: 512 chiều
        ])
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


**`mean_pool` - biến nhiều token thành một vector câu**

Transformer trả embedding cho từng token. Retrieval thì cần một vector đại diện cho cả câu. `attention_mask` giúp bỏ qua padding token để vector câu không bị nhiễu.


In [10]:
def mean_pool(last_hidden_state, attention_mask):
    """
    Gom token embeddings thành một sentence embedding.

    last_hidden_state:
        Tensor shape [batch_size, num_tokens, hidden_dim]
    attention_mask:
        Tensor shape [batch_size, num_tokens]
        Token thật = 1, padding token = 0.

    Vì sao không lấy trung bình thẳng?
        Các câu trong batch có độ dài khác nhau nên tokenizer thêm padding.
        Nếu tính cả padding, vector câu sẽ bị kéo lệch bởi token rỗng.
    """
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    denom = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / denom


**`Stage2StudentProjection` - đóng gói encoder và projection head**

Class này là pipeline inference của ViFashionCLIP text: token ids -> encoder -> mean pooling -> projection. Khi load checkpoint, trọng số encoder và projection sẽ được nạp vào model này.


In [11]:
class Stage2StudentProjection(nn.Module):
    """
    Model inference hoàn chỉnh của ViFashionCLIP text.

    Luồng forward:
        input_ids + attention_mask
          -> student encoder
          -> mean_pool
          -> projection_head
          -> vector 512 chiều

    Tên "Stage2" đến từ quá trình train: student encoder + projection head
    đã được tinh chỉnh để tiệm cận không gian FashionCLIP teacher.
    """

    def __init__(self, encoder, projection_head):
        super().__init__()
        self.encoder = encoder
        self.projection_head = projection_head

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = mean_pool(outputs.last_hidden_state, attention_mask)
        return self.projection_head(pooled)


print("[OK] Neural blocks đã sẵn sàng: ResidualMLPBlock | ProjectionHead | mean_pool | Stage2StudentProjection")


[OK] Neural blocks đã sẵn sàng: ResidualMLPBlock | ProjectionHead | mean_pool | Stage2StudentProjection


## BƯỚC 7: `BGEM3Embeddings` Cho Layer B

Layer B lưu **tri thức stylist**: phong cách, bối cảnh, dáng người, tone da, lý do phối đồ. Đây là text ngôn ngữ tự nhiên, không phải sản phẩm cụ thể.

Vì vậy Layer B dùng BGE-M3 qua Ollama:

```text
query phối đồ -> BGE-M3 vector 1024 chiều -> Qdrant layer_b_female/layer_b_male
```

Không dùng class này cho sản phẩm Layer A, vì collection sản phẩm đang dùng vector 512 chiều từ ViFashionCLIP.


In [12]:
class BGEM3Embeddings(Embeddings):
    """
    LangChain wrapper cho BGE-M3 qua Ollama.

    Dùng khi:
        - Index rule phối đồ vào Layer B.
        - Embed query kiểu "phối đồ đi tiệc cho dáng quả lê" để tìm rule phù hợp.

    Không dùng khi:
        - Tìm sản phẩm thật trong kho. Việc đó thuộc Layer A và dùng ViFashionCLIP.
    """

    def __init__(self, model_name: str = "bge-m3"):
        # OllamaEmbeddings không load model trong Python process này.
        # Nó gửi request sang Ollama server ở localhost:11434.
        self.ollama_embeddings = OllamaEmbeddings(
            model=model_name,
            base_url=OLLAMA_BASE_URL,
        )

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        """Embed nhiều rule stylist khi index Layer B vào Qdrant."""
        return self.ollama_embeddings.embed_documents(texts)

    def embed_query(self, text: str) -> list[float]:
        """Embed một query phối đồ khi cần tìm rule phù hợp."""
        return self.ollama_embeddings.embed_query(text)


print("[OK] BGEM3Embeddings đã được định nghĩa cho Layer B (1024-dim)")


[OK] BGEM3Embeddings đã được định nghĩa cho Layer B (1024-dim)


## BƯỚC 8: `ViFashionCLIPTextEmbeddings` Cho Layer A Text

Đây là class quan trọng nhất cho text retrieval sản phẩm.

Nó làm ba việc:

1. Load checkpoint đã train của student tiếng Việt.
2. Ghép encoder + projection head thành model inference.
3. Encode text sản phẩm hoặc query người dùng thành vector 512 chiều đã L2-normalize.

```text
"tìm áo sơ mi trắng đi làm"
  -> tokenizer
  -> Vietnamese student encoder
  -> mean_pool
  -> ProjectionHead
  -> L2 normalize
  -> vector 512 chiều để search Qdrant
```

Nếu retrieval lấy sai sản phẩm, đây là một trong những nơi cần kiểm tra: checkpoint có đúng không, vector có đúng 512 chiều không, query đưa vào có đúng ý không.


In [13]:
class ViFashionCLIPTextEmbeddings(Embeddings):
    """
    LangChain Embeddings wrapper cho ViFashionCLIP tiếng Việt.

    Dùng cho Layer A text:
        - Embed 65k sản phẩm khi index.
        - Embed query tiếng Việt khi user tìm sản phẩm.

    Output:
        vector 512 chiều, cùng không gian với FashionCLIP teacher.

    Lưu ý tốc độ:
        Khởi tạo class này có thể chậm vì phải load tokenizer, encoder và checkpoint.
        Trong app thật nên lazy-load hoặc dùng remote embedding service.
    """

    def __init__(
        self,
        checkpoint_path: str | Path = VIFASHIONCLIP_CHECKPOINT,
        batch_size: int = 32,
    ):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.batch_size = batch_size
        self.checkpoint_path = Path(checkpoint_path)

        if not self.checkpoint_path.exists():
            raise FileNotFoundError(f"Checkpoint không tồn tại: {self.checkpoint_path}")

        # Checkpoint là nguồn sự thật: nó cho biết student model và teacher_dim đã dùng khi train.
        checkpoint = torch.load(self.checkpoint_path, map_location="cpu")
        student_name = checkpoint.get("student_model_name", STUDENT_MODEL_NAME)
        teacher_dim = int(checkpoint.get("teacher_dim", 512))

        # Tokenizer biến text thành input_ids/attention_mask.
        self.tokenizer = AutoTokenizer.from_pretrained(student_name)

        # Encoder tạo token embeddings tiếng Việt.
        encoder = AutoModel.from_pretrained(student_name)

        # ProjectionHead đưa embedding tiếng Việt sang không gian FashionCLIP.
        projection = ProjectionHead(
            input_dim=encoder.config.hidden_size,
            output_dim=teacher_dim,
            hidden_dim=PROJECTION_HIDDEN_DIM,
            num_layers=PROJECTION_NUM_LAYERS,
            dropout=PROJECTION_DROPOUT,
        )

        # Load đúng trọng số đã train cho cả encoder và projection.
        encoder.load_state_dict(checkpoint["encoder_state_dict"])
        projection.load_state_dict(checkpoint["projection_state_dict"])

        # eval() + requires_grad=False: chỉ inference, không train trong chatbot.
        self.model = Stage2StudentProjection(encoder, projection).to(self.device).eval()
        for parameter in self.model.parameters():
            parameter.requires_grad = False

        self.embedding_dim = teacher_dim
        print(f"[OK] ViFashionCLIP loaded: {self.checkpoint_path}")
        print(f"     Student: {student_name}")
        print(f"     Device : {self.device} | Dim: {self.embedding_dim}")

    @torch.no_grad()
    def _encode(self, texts: list[str]) -> list[list[float]]:
        """
        Encode một batch text thành vector 512 chiều.

        Bước 1: tokenize text.
        Bước 2: chạy model encoder + projection.
        Bước 3: L2 normalize để cosine similarity trong Qdrant ổn định hơn.
        """
        all_embeds = []
        for start in tqdm(range(0, len(texts), self.batch_size), desc="ViFashionCLIP embed", leave=False):
            batch = [str(text) for text in texts[start:start + self.batch_size]]
            inputs = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=STUDENT_MAX_LENGTH,
                return_tensors="pt",
            ).to(self.device)

            embeds = self.model(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
            )
            embeds = F.normalize(embeds, p=2, dim=-1)
            all_embeds.append(embeds.detach().cpu())

        return torch.cat(all_embeds, dim=0).tolist()

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        """Embed nhiều product documents khi index Layer A vào Qdrant."""
        return self._encode(texts)

    def embed_query(self, text: str) -> list[float]:
        """Embed một query tiếng Việt của người dùng khi search sản phẩm."""
        return self._encode([text])[0]


print("[OK] ViFashionCLIPTextEmbeddings đã được định nghĩa cho Layer A text (512-dim)")


[OK] ViFashionCLIPTextEmbeddings đã được định nghĩa cho Layer A text (512-dim)


## BƯỚC 9: `FashionCLIPImageEmbeddings` Cho Layer A Image

Class này dùng image encoder của FashionCLIP gốc để tìm sản phẩm bằng ảnh.

Tư duy cần nhớ:

- Text tiếng Việt dùng student ViFashionCLIP.
- Ảnh dùng FashionCLIP image encoder gốc.
- Cả hai đều trả vector 512 chiều, nhưng phục vụ hai nhánh retrieval khác nhau.

Class này khá nặng vì phải load `CLIPModel` và `CLIPProcessor`. Trong app thật nên chỉ khởi tạo khi user thật sự gửi ảnh.


In [14]:
class FashionCLIPImageEmbeddings:
    """
    FashionCLIP image encoder cho image retrieval.

    Dùng khi:
        - Index ảnh MAIN của sản phẩm vào Qdrant.
        - Encode ảnh user gửi để tìm sản phẩm tương tự.

    Không dùng khi:
        - Tìm rule phối đồ Layer B.
        - Embed query text tiếng Việt.
    """

    def __init__(self, model_name: str = TEACHER_MODEL_NAME, batch_size: int = 32):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.batch_size = batch_size
        self.model_name = model_name

        # Processor chuẩn hóa ảnh đúng format mà FashionCLIP cần.
        self.processor = CLIPProcessor.from_pretrained(model_name)

        # Model tạo image features trong cùng không gian 512 chiều của FashionCLIP.
        self.model = CLIPModel.from_pretrained(model_name).to(self.device).eval()
        for parameter in self.model.parameters():
            parameter.requires_grad = False

        self.embedding_dim = int(getattr(self.model.config, "projection_dim", 512))
        print(f"[OK] FashionCLIP image encoder: {model_name}")
        print(f"     Device: {self.device} | Dim: {self.embedding_dim}")

    @staticmethod
    def _open_image(image_path: str | Path):
        """Mở ảnh và convert sang RGB để xử lý PNG/JPEG nhất quán."""
        return Image.open(image_path).convert("RGB")

    @torch.no_grad()
    def encode_image_paths(self, image_paths: list[str | Path]) -> list[list[float] | None]:
        """
        Encode nhiều ảnh thành vector 512 chiều.

        Nếu một ảnh lỗi hoặc không đọc được, vị trí tương ứng trả về None.
        Cách này giúp pipeline index ảnh bỏ qua ảnh lỗi thay vì crash toàn bộ.
        """
        results: list[list[float] | None] = [None] * len(image_paths)

        for start in tqdm(range(0, len(image_paths), self.batch_size), desc="FashionCLIP image embed", leave=False):
            batch_paths = image_paths[start:start + self.batch_size]
            images = []
            valid_positions = []

            for offset, path in enumerate(batch_paths):
                try:
                    images.append(self._open_image(path))
                    valid_positions.append(start + offset)
                except Exception as exc:
                    print(f"[WARN] Không đọc được ảnh {path}: {exc}")

            if not images:
                continue

            inputs = self.processor(images=images, return_tensors="pt", padding=True).to(self.device) # type: ignore
            embeds = self.model.get_image_features(**inputs)
            if hasattr(embeds, "pooler_output"):
                embeds = embeds.pooler_output
            embeds = F.normalize(embeds, p=2, dim=-1).detach().cpu().tolist()

            for position, vector in zip(valid_positions, embeds):
                results[position] = vector

        return results

    def embed_image(self, image_path: str | Path) -> list[float] | None:
        """Encode một ảnh query của user để search sản phẩm tương tự."""
        return self.encode_image_paths([image_path])[0]


print("[OK] FashionCLIPImageEmbeddings đã được định nghĩa cho Image Retrieval (512-dim)")
print("     Lưu ý: chỉ khởi tạo class này khi cần xử lý ảnh để tránh tốn RAM/GPU.")


[OK] FashionCLIPImageEmbeddings đã được định nghĩa cho Image Retrieval (512-dim)
     Lưu ý: chỉ khởi tạo class này khi cần xử lý ảnh để tránh tốn RAM/GPU.
